In [ ]:
# Braille YOLO Inference Pipeline

## Install Dependencies

```python
!pip install ultralytics opencv-python
```

## Imports

```python
from ultralytics import YOLO
import cv2
import numpy as np
import os
```

## Load Model

```python
MODEL_PATH = "best.pt"

model = YOLO(MODEL_PATH)
```

## Detect Braille Cells

```python
def detect_braille_cells(image_path, conf=0.25):

    results = model.predict(
        source=image_path,
        conf=conf,
        verbose=False
    )

    boxes = results[0].boxes.xyxy.cpu().numpy()

    return boxes
```

## Sort Cells Into Reading Order

```python
def sort_boxes_reading_order(boxes):

    boxes = sorted(boxes, key=lambda b: b[1])

    heights = [b[3] - b[1] for b in boxes]

    avg_height = np.mean(heights)

    row_threshold = avg_height * 0.7

    rows = []

    for box in boxes:

        if not rows:
            rows.append([box])
            continue

        current_y = box[1]
        row_y = rows[-1][0][1]

        if abs(current_y - row_y) < row_threshold:
            rows[-1].append(box)
        else:
            rows.append([box])

    for row in rows:
        row.sort(key=lambda b: b[0])

    ordered_boxes = []

    for row in rows:
        ordered_boxes.extend(row)

    return ordered_boxes
```

## Crop Cells

```python
def crop_cells(image_path, ordered_boxes):

    image = cv2.imread(image_path)

    crops = []

    for box in ordered_boxes:

        x1, y1, x2, y2 = map(int, box)

        crop = image[y1:y2, x1:x2]

        crops.append(crop)

    return crops
```

## Main Function

```python
def run_vision(image_path):

    boxes = detect_braille_cells(image_path)

    ordered_boxes = sort_boxes_reading_order(boxes)

    crops = crop_cells(image_path, ordered_boxes)

    return crops
```

## Example Usage

```python
image_path = "test.jpg"

crops = run_vision(image_path)

print(f"Detected {len(crops)} braille cells")
```

## Save Crops For Debugging

```python
os.makedirs("cropped_cells", exist_ok=True)

for i, crop in enumerate(crops):

    cv2.imwrite(
        f"cropped_cells/cell_{i:03d}.png",
        crop
    )
```

## Integration With Extractor

Replace:

```python
patterns = []

for crop in crops:

    pattern = extract_cell(crop)

    patterns.append(pattern)

print(patterns)
```

Expected output:

```python
[
    "100000",
    "110000",
    "100100",
    ...
]
```

Then:

```python
text = translate_braille(patterns)

print(text)
```
